In [1]:
%cd ../

/u02/thanh/workplace/plasma


In [2]:
import inspect
import plasma.data_model as pdm

from pydantic import BaseModel
from typing import NamedTuple, dataclass_transform

from plasma.data_model.base_model.field import Field, Composite, List

In [3]:
@pdm.model
class A:
    a:int
    b:int
    c:list[str]

In [4]:
@pdm.model
class B:
    b1:A
    b2:list[A]
    b3:list

In [5]:
@pdm.model
class C:
    c1:B
    c2:list[B]

In [14]:
c = C(
        B(
            A(5, 6, ['a', 'b']),
            [
                A(5, 6, ['a', 'b']),
            ],
            [1, 2]
        ),
        [
            B(
                A(5, 6, ['a', 'b']),
                [],
                [1, 3]
            ),
            B(
                A(5, 6, ['a', 'b']),
                [
                    A(7, 8, ['a', 'b']),
                ],
                [1, 3]
            ),
        ]
    )

c

C
  |-c1=B
    |-b1=A
      |-a=5
      |-b=6
      |-c=['a', 'b']
    |-b2:
      A
        |-a=5
        |-b=6
        |-c=['a', 'b']
    |-b3=[1, 2]
  |-c2:
    B
      |-b1=A
        |-a=5
        |-b=6
        |-c=['a', 'b']
      |-b2=[]
      |-b3=[1, 3]
    B
      |-b1=A
        |-a=5
        |-b=6
        |-c=['a', 'b']
      |-b2:
        A
          |-a=7
          |-b=8
          |-c=['a', 'b']
      |-b3=[1, 3]

In [12]:
class AccessorState(dict[str, object]):
    
    def __init__(self, obj):
        super().__init__()

        for a in pdm.type_inquirer.accessors(type(obj)):
            self.__update(obj, None, a.split('.'))

    def __update(self, obj, key, path):
        if len(path) == 0:
            self[key] = obj
        elif path[0] == '@idx':
            assert isinstance(obj, (tuple, list)), f'expect list or tuple at {key}'
            for i, v in enumerate(obj):
                self.__update(v, f'{key}.{i}', path[1:])
        else:
            next_key = f'{key}.{path[0]}' if key is not None else path[0]
            if obj is None:
                value = None
            else:
                assert hasattr(obj, path[0]), f'no attribute {path[0]} for {obj}'
                value = getattr(obj, path[0])

            self.__update(value, next_key, path[1:])

ac = AccessorState(c)
ac

{'c1.b1.a': 5,
 'c1.b1.b': 6,
 'c1.b1.c': ['a', 'b'],
 'c1.b2.0.a': 5,
 'c1.b2.0.b': 6,
 'c1.b2.0.c': ['a', 'b'],
 'c1.b3': [1, 2],
 'c2.0.b1.a': 5,
 'c2.0.b1.b': 6,
 'c2.0.b1.c': ['a', 'b'],
 'c2.0.b3': [1, 3]}

In [21]:
import networkx as nx

graph = nx.DiGraph()
roots = set()
for k, v in ac.items():
    components = k.split('.')
    paths = [tuple(components[:i + 1]) for i in range(len(components))]
    nx.add_path(graph, paths, value=None)
    graph.add_node(paths[-1], value=v)
    roots.add(paths[0])

In [24]:
C.__struct

{'c1': {'b1': {'a': 'c1.b1.a', 'b': 'c1.b1.b', 'c': 'c1.b1.c'},
  'b2': [{'a': 'c1.b2.@idx.a', 'b': 'c1.b2.@idx.b', 'c': 'c1.b2.@idx.c'}],
  'b3': 'c1.b3'},
 'c2': [{'b1': {'a': 'c2.@idx.b1.a', 'b': 'c2.@idx.b1.b', 'c': 'c2.@idx.b1.c'},
   'b2': [{'a': 'c2.@idx.b2.@idx.a',
     'b': 'c2.@idx.b2.@idx.b',
     'c': 'c2.@idx.b2.@idx.c'}],
   'b3': 'c2.@idx.b3'}]}

In [ ]:
def resolve(graph:nx.DiGraph, node, state):
    if graph.out_degree(node) > 0:
        for s in graph.successors(node):
            resolve(graph, s, state)
        
        
    pass

In [23]:
graph.nodes

NodeView((('c1',), ('c1', 'b1'), ('c1', 'b1', 'a'), ('c1', 'b1', 'b'), ('c1', 'b1', 'c'), ('c1', 'b2'), ('c1', 'b2', '0'), ('c1', 'b2', '0', 'a'), ('c1', 'b2', '0', 'b'), ('c1', 'b2', '0', 'c'), ('c1', 'b3'), ('c2',), ('c2', '0'), ('c2', '0', 'b1'), ('c2', '0', 'b1', 'a'), ('c2', '0', 'b1', 'b'), ('c2', '0', 'b1', 'c'), ('c2', '0', 'b3')))

In [8]:
class StructState(dict):
    
    def __init__(self, obj):
        super().__init__()

        if obj is not None:
            struct = pdm.type_inquirer.struct(type(obj))
            for k in struct:
                self.__update(struct, k, obj)
        
    def __update(self, template, key, obj):
        next_template = template[key]
        value = getattr(obj, key)
        if isinstance(next_template, dict):
            self[key] = StructState(value)
        elif isinstance(next_template, list):
            assert isinstance(value, (list, tuple)), f'expected list or tuple for {obj} at {key}'
            self[key] = [StructState(v) for v in value]
        else:
            self[key] = value
            
sc = StructState(c)
sc

{'c1': {'b1': {'a': 5, 'b': 6, 'c': ['a', 'b']},
  'b2': [{'a': 5, 'b': 6, 'c': ['a', 'b']}],
  'b3': [1, 2]},
 'c2': [{'b1': {'a': 5, 'b': 6, 'c': ['a', 'b']},
   'b2': [{'a': 5, 'b': 6, 'c': ['a', 'b']},
    {'a': 5, 'b': 7, 'c': ['a', 'b']}],
   'b3': [1, 3]}]}

In [ ]:
class Accessor2Struct(dict):
    
    def __init__(self, accessors:dict[str, object]):
        super().__init__()
        
    def __update(self, key, value):
        pass


In [9]:
C.__accessors

{'c1.b1.a': C.c1.b1.a,
 'c1.b1.b': C.c1.b1.b,
 'c1.b1.c': C.c1.b1.c,
 'c1.b2.@idx.a': A.a,
 'c1.b2.@idx.b': A.b,
 'c1.b2.@idx.c': A.c,
 'c1.b3': C.c1.b3,
 'c2.@idx.b1.a': B.b1.a,
 'c2.@idx.b1.b': B.b1.b,
 'c2.@idx.b1.c': B.b1.c,
 'c2.@idx.b2.@idx.a': A.a,
 'c2.@idx.b2.@idx.b': A.b,
 'c2.@idx.b2.@idx.c': A.c,
 'c2.@idx.b3': B.b3}

In [10]:
C.__struct

{'c1': {'b1': {'a': 'c1.b1.a', 'b': 'c1.b1.b', 'c': 'c1.b1.c'},
  'b2': [{'a': 'c1.b2.@idx.a', 'b': 'c1.b2.@idx.b', 'c': 'c1.b2.@idx.c'}],
  'b3': 'c1.b3'},
 'c2': [{'b1': {'a': 'c2.@idx.b1.a', 'b': 'c2.@idx.b1.b', 'c': 'c2.@idx.b1.c'},
   'b2': [{'a': 'c2.@idx.b2.@idx.a',
     'b': 'c2.@idx.b2.@idx.b',
     'c': 'c2.@idx.b2.@idx.c'}],
   'b3': 'c2.@idx.b3'}]}